In [3]:
!pip install -q langchain-classic langchain_openai langchain-core langchain-community langgraph langchain-tavily youtube-search

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [2]:
import os

with open("/content/api_key_openia.txt", "r", encoding="utf-8-sig") as archivo:
  apikey = archivo.readline().strip()
  os.environ["OPENAI_API_KEY"] = apikey

with open("/content/api_tavily.txt", "r", encoding="utf-8-sig") as archivo:
  apikey = archivo.readline().strip()
  os.environ["TAVILY_API_KEY"] = apikey

# **Herramienta 1: Idenficador de gaps de conocimiento**//
Tool 1: Skill Gap Analyzer

In [4]:
from typing import Optional
from langchain_core.tools import BaseTool
from pydantic import BaseModel, Field
from langchain_core.callbacks import (AsyncCallbackManagerForToolRun, CallbackManagerForToolRun)

# Definición de los tipos de campo:
class SkillGapInput(BaseModel):
    perfil_actual: str = Field(description="Habilidades y formación actual del usuario")
    objetivo_hacer: str = Field(description="La capacidad o proyecto que el usuario desea lograr")

class SkillGapTool(BaseTool):
    name: str = "skill_gap_analyzer"
    description: str = "Analiza qué habilidades técnicas faltan para lograr un objetivo específico."
    args_schema: type[BaseModel] = SkillGapInput

    def _run(self, perfil_actual: str, objetivo_hacer: str, run_manager: Optional[CallbackManagerForToolRun] = None) -> str:
        """Ejecución síncrona"""
        return f"Analiza como experto qué conceptos técnicos le faltan a un {perfil_actual} para hacer {objetivo_hacer}. Sé específico con librerías y conceptos de IA."

    async def _arun(self, perfil_actual: str, objetivo_hacer: str, run_manager: Optional[AsyncCallbackManagerForToolRun] = None) -> str:
        """Ejecución asíncrona"""
        return self._run(perfil_actual, objetivo_hacer, run_manager=run_manager.get_sync())

# **Herramienta 2: Idenficador de Recursos y Expertos**


Tool 2: Deep Expert Hunter

In [5]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults

# Definición de clases
class SearchInput(BaseModel):
    tema: str = Field(description="El tema o tecnología a investigar en la web")

# Clase de la Herramienta
class DeepExpertHunter(BaseTool):
    name: str = "deep_expert_hunter"
    description: str = "Busca en la web y resume los 3 mejores recursos o tutoriales sobre un tema."
    args_schema: type[BaseModel] = SearchInput

    def _run(self, tema: str, run_manager: Optional[CallbackManagerForToolRun] = None) -> str:
        search = TavilySearchResults(max_results=5)
        resultados = search.run(tema)

        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
        parser = StrOutputParser()

        prompt = ChatPromptTemplate.from_messages([
            ("system", "Eres un analista de formación tech. Resume los 3 mejores enlaces de estos resultados para una persona no experta aprendiendo sobre tecnología."),
            ("human", "Resultados para '{tema}':\n\n{resultados}")
        ])

        # Elaboración de cadena
        chain = prompt | llm | parser
        return chain.invoke({"tema": tema, "resultados": resultados})

    async def _arun(self, tema: str, run_manager: Optional[AsyncCallbackManagerForToolRun] = None) -> str:
        return self._run(tema, run_manager=run_manager.get_sync())

# **Herramienta 3: Idenficador de tutoriales**


Uso de herramienta externa: YouTube Search Tool

In [6]:
from langchain_community.tools import YouTubeSearchTool

class YouTubeInput(BaseModel):
    query: str = Field(description="El tema o tutorial específico a buscar en YouTube")

class YouTubeHunterTool(BaseTool):
    name: str = "youtube_hunter"
    description: str = "Busca videos y tutoriales en YouTube sobre temas tecnológicos específicos."
    args_schema: type[BaseModel] = YouTubeInput

    def _run(self, query: str, run_manager: Optional[CallbackManagerForToolRun] = None) -> str:
        """Búsqueda en YouTube."""
        # Herramienta externa: API Youtube
        yt_search = YouTubeSearchTool()

        # Número de resultados
        resultados = yt_search.run(f"{query},3")

        return f"Aquí tienes los videos más relevantes encontrados en YouTube para '{query}':\n{resultados}"

    async def _arun(self, query: str, run_manager: Optional[AsyncCallbackManagerForToolRun] = None) -> str:
        """Ejecución asíncrona."""
        return self._run(query, run_manager=run_manager.get_sync())

**Herramienta 4: Ideas de proyectos**


In [7]:
class ProjectInput(BaseModel):
    idea_proyecto: str = Field(description="Una idea creativa de proyecto que una el perfil del usuario con la tecnología")
    tecnologia_clave: str = Field(description="La tecnología que el usuario acaba de aprender")
    perfil_usuario: str = Field(description="El perfil del usuario")

class ProjectArchitectTool(BaseTool):
    name: str = "generador_proyectos"
    description: str = "Diseña una idea de proyecto práctico. Debes proponer una idea creativa basada en el perfil."
    args_schema: type[BaseModel] = ProjectInput

    def _run(self, idea_proyecto: str, tecnologia_clave: str, perfil_usuario: str, run_manager: Optional[CallbackManagerForToolRun] = None) -> str:
        """Lógica para proponer un proyecto dinámico."""
        return f"""
        Propuesta de Proyecto para un {perfil_usuario}:
        ---
        Idea: {idea_proyecto}
        Descripción: Un sistema basado en {tecnologia_clave} diseñado para potenciar el perfil de {perfil_usuario}.
        Valor: Este proyecto demuestra que puedes se puede aplicar lo aprendido en tu campo profesional u objetivo deseado.
        """

**Set de 4 herramientas**

In [8]:
gap_analyzer = SkillGapTool()
resource_hunter = DeepExpertHunter()
tutorial_finder = YouTubeHunterTool()
project_architect = ProjectArchitectTool()

toolkit = [gap_analyzer, resource_hunter, tutorial_finder, project_architect]

**Creación de agente**

In [9]:
from langgraph.checkpoint.memory import MemorySaver
from langchain.agents import create_agent

llm = ChatOpenAI(temperature=0)
memory = MemorySaver()

system_prompt = """Eres el 'Mentor Inteligente de Conocimientos de IA'. Tu objetivo es ayudar a los usuarios
                  a profundizar su conocimiento de IA mediante un flujo de trabajo de 4 pasos:

                  1. Identificar qué habilidades les faltan (Usa: skill_gap_analyzer).
                  2. Buscar fuentes y documentación en la web (Usa: deep_expert_hunter).
                  3. Encontrar tutoriales prácticos en video (Usa: identificador_tutoriales).
                  4. Realiza una propuesta de un proyecto innovador de acuerdo al perfil (Usa: generador_proyectos).

                  Sé directo, profesional y proporciona siempre los enlaces encontrados por las herramientas."""

agent =create_agent(model=llm,tools=toolkit,system_prompt=system_prompt,checkpointer=memory)


**Prueba de agente**

In [10]:
from langchain_core.messages import HumanMessage

# Configuración del hilo para la memoria
config = {"configurable": {"thread_id": "prueba2"}}

# Prompt de prueba #1
input_message = {"messages": [HumanMessage(
    content=
    "Hola, soy economista y quiero aprender a crear agentes de IA para análisis financiero. "
     "Analiza mi brecha, busca recursos de lectura y dame una idea de proyecto.")]}

# Ejecución en streaming para ver el razonamiento paso a paso
for step in agent.stream(input_message, config, stream_mode="values"):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Hola, soy economista y quiero aprender a crear agentes de IA para análisis financiero. Analiza mi brecha, busca recursos de lectura y dame una idea de proyecto.
================================== Ai Message ==================================
Tool Calls:
  skill_gap_analyzer (call_bF4uLVFHzRhxKNG710bypz9X)
 Call ID: call_bF4uLVFHzRhxKNG710bypz9X
  Args:
    perfil_actual: Economista
    objetivo_hacer: Crear agentes de IA para análisis financiero
================================= Tool Message =================================
Name: skill_gap_analyzer

Analiza como experto qué conceptos técnicos le faltan a un Economista para hacer Crear agentes de IA para análisis financiero. Sé específico con librerías y conceptos de IA.
================================== Ai Message ==================================
Tool Calls:
  deep_expert_hunter (call_Xoa1NLxJpaNspLDCMds7V4bV)
 Call ID: call_Xoa1NLxJpaNspLDCMds7V4bV
 

/tmp/ipython-input-1537837107.py:17: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search = TavilySearchResults(max_results=5)


KeyboardInterrupt: 

## **Uso de RAG**

In [17]:
!pip install pypdf
!pip install -qU unstructured

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.3/331.3 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 14.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.8/453.8 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.8/167.8 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 94.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.2/220.2 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 5.5 MB/s eta 0:00:00
 

In [64]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

# Lista donde se guardan todos los documentos
all_docs = []

#PDFs
loader_pdf1 = PyPDFLoader(file_path="/content/Unesco.pdf")
all_docs.extend(loader_pdf1.load())

loader_pdf2 = PyPDFLoader(file_path="/content/WEF_Future_of_Jobs_Report_2025.pdf")
all_docs.extend(loader_pdf2.load())

text_splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=100)
all_splits = text_splitter.split_documents(all_docs)


Técnica de Hyde

In [51]:
from langchain_openai import OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.hyde.base import HypotheticalDocumentEmbedder

base_embeddings = OpenAIEmbeddings()

custom_prompt = PromptTemplate.from_template("""
Eres un experto en el futuro del trabajo y marcos de competencias de la UNESCO y el World Economic Forum.
Para la siguiente duda, redacta un párrafo técnico que describa las habilidades necesarias:
Pregunta: {QUESTION}
Respuesta técnica hipotética:
""")

# Crear instancia de HyDE
hyde_embeddings = HypotheticalDocumentEmbedder.from_llm(llm, base_embeddings, custom_prompt=custom_prompt)

In [65]:
vector_store = InMemoryVectorStore(hyde_embeddings)
vector_store.add_documents(documents=all_splits)

['24631d9e-ac33-4ca5-8e36-c7371469ee4c',
 '754b9508-272a-445d-b40d-c637ac104221',
 'a55166a6-66bd-4468-9b00-e62e08c9557f',
 '5d1a1588-ab32-49f0-9070-53e8a0f4ab36',
 'e7f1feb4-6e40-48a9-b9a7-42b774086125',
 '886e2ee2-b5db-4205-947e-1a5c84321e8b',
 'e0429289-f16d-4260-bd40-23cc954b3fe9',
 '1e6357c2-7215-4f28-912c-db7d4f7f114e',
 'a8524f8b-33f7-4ea6-a25e-304bd74a4ed3',
 '0e25f9c0-e2ab-497c-9aa6-ad3dc31607cd',
 '188308d0-b7e7-4eef-b872-2c377321d500',
 '9453ab47-646a-4130-98dd-d4874a5a07f4',
 '89360b77-f52f-471f-bc1f-b232f5639316',
 'dfef3c79-1a52-4695-a175-87c276d6ef74',
 'e91638d9-2f7f-4b14-bedb-01845de8e32c',
 'bfc6e266-6551-4eca-bfd2-8632e59bb3d6',
 '2f70796e-7815-4065-a0f3-478ee8c6b0f7',
 'b1964902-9abf-47f5-ad95-912371d25b00',
 'a8d8f110-9e41-41c2-bcd1-b9926ba314a1',
 '43f51de8-894d-4050-8c58-71f12ad4d250',
 'a18cc31a-67f4-42ca-8ed6-c0e55f0fa74c',
 '4c064c11-e7ee-425a-ab3a-a8f6e73ac0d4',
 '520800aa-6771-4da9-b84b-82cf93842c9f',
 'f701d24e-a311-4e85-9e44-6f56b4950d99',
 '40c43786-75f0-

Herramienta 5: Recuperador de Conocimiento Experto (con RAG)

In [66]:
# Esquema de entrada
from typing import Optional

class RAGInput(BaseModel):
    consulta: str = Field(description="El tema técnico o competencia a buscar en los documentos.")
    # El agente usará este campo para filtrar si el usuario menciona UNESCO o WEF
    fuente_filtro: Optional[str] = Field(default=None, description="Filtro opcional: 'UNESCO' o 'WEF'")

class ExpertKnowledgeTool(BaseTool):
    name: str = "expert_knowledge_retriever"
    description: str = "Busca en documentos oficiales. Permite filtrar por UNESCO o WEF si es necesario."
    args_schema: type[BaseModel] = RAGInput

    def _run(self, consulta: str, fuente_filtro: Optional[str] = None) -> str:
        # Lógica de filtrado de metadatos
        filtro_metadata = None
        if fuente_filtro:
            if "UNESCO" in fuente_filtro.upper():
                # Filtramos por el nombre del archivo de la UNESCO
                filtro_metadata = lambda doc: "Unesco.pdf" in doc.metadata.get("source", "")
            elif "WEF" in fuente_filtro.upper():
                # Filtramos por el nombre del archivo del WEF
                filtro_metadata = lambda doc: "WEF_Future_of_Jobs_Report_2025.pdf" in doc.metadata.get("source", "")

        # Ejecución de la búsqueda con el filtro incorporado
        # El vector_store (que ya tiene HyDE) aplicará el filtro de metadatos aquí
        docs = vector_store.similarity_search(consulta, k=3, filter=filtro_metadata)

        if not docs:
            return "No se encontró información específica en la fuente solicitada."

        resultados = []
        for doc in docs:
            # Extraemos el nombre del archivo
            fuente = doc.metadata.get('source', 'Documento').split('/')[-1]
            # Extraemos el número de página (sumamos 1 porque empieza en 0)
            pagina = doc.metadata.get('page', 0) + 1

            # CORRECCIÓN: Usamos 'pagina' en lugar de 'p'
            fragmento = f"--- FUENTE: {fuente} (Pág. {pagina}) ---\n{doc.page_content[:400]}..."
            resultados.append(fragmento)

        return "Información oficial recuperada:\n\n" + "\n\n".join(resultados)

expert_retriever = ExpertKnowledgeTool()

In [67]:
#Set de herramientas actualizado

toolkit = [gap_analyzer, resource_hunter, tutorial_finder, project_architect, expert_retriever]

In [68]:
# Nuevo system prompt (Uso de RAG)

system_prompt = """Eres el 'Mentor Inteligente de Conocimientos de IA'. Tu objetivo es ayudar a los usuarios
                  a profundizar su conocimiento mediante un flujo de trabajo de 5 pasos obligatorios:

                  1. **Analizar Brechas**: Identifica habilidades faltantes con 'skill_gap_analyzer'.
                  2. **Consultar Fuentes Oficiales (RAG)**: Usa 'expert_knowledge_retriever'.
                     *Si el usuario pide específicamente la UNESCO o el WEF, usa el parámetro 'fuente_filtro'.* DEBES citar siempre el nombre del archivo y la página exacta.
                  3. **Investigar Tendencias**: Usa 'deep_expert_hunter' para buscar documentación web reciente.
                  4. **Localizar Tutoriales**: Encuentra videos prácticos con 'youtube_hunter'.
                  5. **Diseñar Proyecto**: Propón un proyecto con 'generador_proyectos' basado en la info recuperada.

                  Responde siempre de forma estructurada y profesional."""

# 3. Creación del agente con el toolkit completo (que ya incluye a expert_retriever)
agent = create_agent(
    model=llm,
    tools=toolkit,
    system_prompt=system_prompt,
    checkpointer=memory
)

Prueba de agente con herramientas actualizadas

In [69]:
# Configuración del hilo para la memoria
config = {"configurable": {"thread_id": "prueba_1"}}

# Prompt de prueba #1
input_message = {"messages": [HumanMessage(
    content="""
Hola, soy un Analista Financiero y quiero especializarme en IA aplicada a la predicción de mercados. Por favor:
1. Analiza qué me falta aprender.
2. Dime qué dicen los manuales de la UNESCO y el WEF sobre las competencias que necesito (cita páginas).
3. Busca artículos técnicos recientes en la web.
4. Encuéntrame tutoriales prácticos en YouTube.
5. Diséñame un proyecto innovador para mi portafolio.
"""
     )]}

# Ejecución en streaming para ver el razonamiento paso a paso
for step in agent.stream(input_message, config, stream_mode="values"):
    step["messages"][-1].pretty_print()

================================ Human Message =================================


Hola, soy un Analista Financiero y quiero especializarme en IA aplicada a la predicción de mercados. Por favor:
1. Analiza qué me falta aprender.
2. Dime qué dicen los manuales de la UNESCO y el WEF sobre las competencias que necesito (cita páginas).
3. Busca artículos técnicos recientes en la web.
4. Encuéntrame tutoriales prácticos en YouTube.
5. Diséñame un proyecto innovador para mi portafolio.

================================== Ai Message ==================================
Tool Calls:
  skill_gap_analyzer (call_ovmgpvTZCqRbSRbenGsYaivA)
 Call ID: call_ovmgpvTZCqRbSRbenGsYaivA
  Args:
    perfil_actual: Analista Financiero
    objetivo_hacer: IA aplicada a la predicción de mercados
================================= Tool Message =================================
Name: skill_gap_analyzer

Analiza como experto qué conceptos técnicos le faltan a un Analista Financiero para hacer IA aplicada a la predicc